# 🛡️ Detector UEBA: Detecção de Anomalias em Logs de Autenticação
### Dissertação de Mestrado Profissional em Engenharia Elétrica (UnB / FT / ENE)
* **Autor:** Allysson de Oliveira Noronha
* **Orientador:** Prof. Dr. Robson de Oliveira Albuquerque
* **Repositório:** [https://github.com/allyssonnor/anti_Detector](https://github.com/allyssonnor/anti_Detector)

---

### 🎯 Propósito deste Demonstrador
Este notebook foi elaborado para que a **Banca Examinadora** possa reproduzir e auditar em minutos o pipeline experimental:
1. **Auditoria de Causalidade (`pytest`):** Garantia matemática e algorítmica de ausência de vazamento temporal (*data leakage*).
2. **Treinamento e Calibração de Limiares:** Treino cronológico no baseline (`data/seed_1_sample.jsonl`) com otimização dinâmica de limiares operacionais via $F_2$-Score na validação.
3. **Inferência Cruzada e Generalização:** Aplicação do modelo treinado sobre o dataset de teste independente (`data/ef_amplo.jsonl` ou `data/test_bruteforce_42.jsonl`).
4. **Dashboard Forense e Métricas:** Visualização imediata de Matriz de Confusão, PR-AUC, ROC-AUC, Lift e explicabilidade dos eventos detectados.

## 1. ⚙️ Preparação do Ambiente e Clonagem do Repositório

In [ ]:
# 1. Clonagem do repositório
import os

REPO_URL = "https://github.com/allyssonnor/anti_Detector.git"

if not os.path.exists("anti_Detector"):
    !git clone $REPO_URL
    %cd anti_Detector
else:
    %cd anti_Detector
    !git pull

# 2. Instalação das dependências necessárias
!pip install -q -r requirements.txt

## 2. 🧪 Auditoria de Integridade e Causalidade Temporal
Execução da suíte de testes de causalidade para comprovar que nenhuma feature utiliza informação futura no instante $t$.

In [ ]:
!python -m pytest tests/test_causality.py -v

## 3. 🏋️ Treinamento e Calibração Dinâmica de Limiares
Execução do protocolo de treino anti-leakage com particionamento cronológico (70% treino, 20% validação, 10% teste misto), contemplando RandomForest e Autoencoder MLP.

In [ ]:
!python scripts/detectar.py --exp_config config/demo_colab.yaml

## 4. 📊 Resultados Comparativos do Treino e Teste Misto

In [ ]:
import pandas as pd
import glob

res_csv = glob.glob("results/demo_colab/**/master_results.csv", recursive=True)
if res_csv:
    df_res = pd.read_csv(res_csv[0])
    cols = [c for c in ["model", "split", "precision", "recall", "f1_score", "f2_score", "pr_auc", "roc_auc", "lift"] if c in df_res.columns]
    print("\n📈 TABELA CONSOLIDADA DE RESULTADOS:")
    display(df_res[cols].drop_duplicates())
else:
    print("Nenhum resultado consolidado encontrado.")

## 5. 🔍 Inferência Cruzada em Cenário Independente
Aplicação do modelo treinado e dos limiares calibrados em dados nunca vistos (`data/ef_amplo.jsonl` ou `data/test_bruteforce_42.jsonl`), avaliando a capacidade de generalização e explicabilidade.

In [ ]:
# Modelo RandomForest gerado no treino anterior
MODEL_DIR = "results/demo_colab/full/run_seed42/Synthetic_Chronological_RandomForest_global_20"
DATA_TARGET = "data/ef_amplo.jsonl"

!python scripts/inferir_usual.py --config config/demo_colab.yaml --data $DATA_TARGET --model_dir $MODEL_DIR

## 6. 🎨 Dashboard Forense e Explicabilidade dos Alertas

In [ ]:
from IPython.display import Image, display
import os

dashboard_img = f"{MODEL_DIR}/dashboard_ef_amplo.png"
if os.path.exists(dashboard_img):
    display(Image(dashboard_img))
else:
    print(f"Dashboard não encontrado em: {dashboard_img}")

# Amostra das detecções e anomalias identificadas
eventos_csv = f"{MODEL_DIR}/inferencia_ef_amplo.csv"
if os.path.exists(eventos_csv):
    df_inf = pd.read_csv(eventos_csv)
    print("\n🛡️ Top 10 Alertas Disparados com Detalhes Comportamentais:")
    alertas = df_inf[df_inf["alert"] == 1] if "alert" in df_inf.columns else df_inf.head(10)
    cols_show = [c for c in ["UserID", "Time", "IpAddress", "LogHost", "anomaly_score", "threshold", "alert", "Is_Anomaly", "AnomalyType"] if c in df_inf.columns]
    display(alertas[cols_show].head(10))